# Practical 1 · Magnitude prediction with machine learning

Compare linear regression, a dense neural network (ANN), and a one-dimensional convolutional neural network (CNN). Explore K-means clustering of engineered features and waveforms.

The lesson generates **800 synthetic waveforms** with simplified P- and S-wave signals. No external dataset or trained model is required. This is a worked teaching example: accuracy on this generator does not establish performance on real earthquake recordings.

## Running the lesson

Run the setup cell first, then run the remaining cells in order. A CPU works; an available GPU can speed up CNN training. In Colab you can select a GPU under **Runtime → Change runtime type** before running the lesson. Set `MAX_EPOCHS` lower for a shorter demonstration.

Use **File → Save a copy in Drive** to keep notebook edits. Download generated PNG figures if you want to retain them after the runtime ends.


In [ ]:
# Run first in Colab; this fetches Utility.py and the dependency list.
import os
import sys
import subprocess
from pathlib import Path

LESSON = "practical_01_magnitude_machine_learning"
if "google.colab" in sys.modules:
    repo_dir = Path("/content/applied_seismology_rwth_practical_01")
    if not repo_dir.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/sebmagia/applied_seismology_rwth.git",
            str(repo_dir),
        ], check=True)
    lesson_dir = repo_dir / LESSON
    if not (lesson_dir / "Utility.py").is_file():
        raise FileNotFoundError(
            "Upload the complete lesson folder to GitHub, then start a fresh Colab runtime."
        )
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(lesson_dir / "requirements.txt"),
    ], check=True)
    os.chdir(lesson_dir)
else:
    if (Path.cwd() / LESSON).is_dir():
        os.chdir(Path.cwd() / LESSON)
    if not Path("Utility.py").is_file():
        raise FileNotFoundError("Open this notebook from the lesson folder containing Utility.py.")

# Ensure imports resolve after changing directories in a notebook runtime.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print("Lesson directory:", Path.cwd())


In [ ]:

# Import libraries and utilities
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

# Import our custom utilities
from Utility import *

# Set plotting style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully!")

# Lesson settings: reduce MAX_EPOCHS for a shorter demonstration.
N_SAMPLES = 800
MAX_EPOCHS = 100
BATCH_SIZE = 32
RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))


## 1. Generate and inspect synthetic waveforms

In [ ]:
# Generate synthetic earthquake dataset
print("\nGenerating synthetic earthquake waveforms...")
print("This simulates real earthquake recordings with P-waves and S-waves")

# Generate dataset
waveforms, magnitudes, distances = generate_dataset(n_samples=N_SAMPLES, random_seed=RANDOM_SEED)

# Display basic statistics
print(f"\nDataset Summary:")
print(f"- Number of waveforms: {len(waveforms)}")
print(f"- Waveform length: {len(waveforms[0])} samples")
print(f"- Duration: 60 seconds at 100 Hz sampling rate")
print(f"- Magnitude range: {magnitudes.min():.2f} - {magnitudes.max():.2f}")
print(f"- Distance range: {distances.min():.1f} - {distances.max():.1f} km")

In [ ]:
# Display example waveforms
print("\nVisualizing example waveforms:")
print("Notice how larger magnitudes have higher amplitudes")
print("P-waves arrive first (higher frequency), followed by S-waves (lower frequency)")

# Plot example waveforms
plot_waveform_examples(waveforms, magnitudes, distances, n_examples=4)

## 2. Extract waveform features

In [ ]:
# Feature extraction
print("\nExtracting features from raw waveforms...")
print("Converting time-series data into meaningful features for ML")

# Extract features
features = extract_features(waveforms)
feature_names = get_feature_names()

print(f"\nFeature extraction complete:")
print(f"- Feature matrix shape: {features.shape}")
print(f"- Features per waveform: {features.shape[1]}")
print("\nExtracted features:")
for i, name in enumerate(feature_names):
    print(f"  {i+1}. {name}")

# Feature importance analysis
print("\nAnalyzing which features are most important for magnitude prediction...")

# Analyze feature importance
# Correlations are evaluated on the training subset after splitting below.

## 3. Separate training, validation and test events

Fit scalers on training events only. The validation events guide early stopping; the test events are reserved for final evaluation. All three supervised models use the same split.

In [ ]:
# Split event indices before fitting any scalers.
# Reuse the same events for the feature models and the waveform CNN.
indices = np.arange(len(magnitudes))
train_val_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_SEED
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.2, random_state=RANDOM_SEED
)

feature_scaler = StandardScaler().fit(features[train_idx])
target_scaler = StandardScaler().fit(magnitudes[train_idx, None])
features_scaled = feature_scaler.transform(features)
magnitudes_scaled = target_scaler.transform(magnitudes[:, None])

X_train, X_val, X_test = (features_scaled[idx] for idx in (train_idx, val_idx, test_idx))
y_train, y_val, y_test = (magnitudes_scaled[idx] for idx in (train_idx, val_idx, test_idx))

print(f"Training: {len(train_idx)}, validation: {len(val_idx)}, test: {len(test_idx)}")
print("Scalers are fitted on training events only.")
analyze_feature_importance(features[train_idx], magnitudes[train_idx], feature_names)


## 4. Linear regression baseline

In [ ]:
# Linear regression
from sklearn.linear_model import LinearRegression

reg = LinearRegression().fit(X_train, y_train)

y_pred = reg.predict(X_test)

y_pred_invers = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
y_test_invers = target_scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculate evaluation metrics
metrics = evaluate_model(y_test_invers, y_pred_invers)

print(f"Test Set Performance:")
print(f"- Mean Squared Error (MSE): {metrics['mse']:.4f}")
print(f"- Root Mean Squared Error (RMSE): {metrics['rmse']:.4f}")
print(f"- Mean Absolute Error (MAE): {metrics['mae']:.4f}")
print(f"- R² Score: {metrics['r2']:.4f}")

# Plot prediction results
plot_prediction_results(y_test_invers, y_pred_invers, "Linear Regression Magnitude Prediction", save_title = 'Linear_Regression')

# Get coefficients and intercept
intercept = reg.intercept_
intercept = intercept[0]
coefficients = reg.coef_
coefficients = coefficients[0]

# Create the equation as a string
equation = f"Standardized magnitude = {intercept:.4f}\n"
for coef, name in zip(coefficients, feature_names):
    equation += f"+ ({coef:.4f} * {name})\n"

print("Coefficients below apply to standardized input features and standardized magnitude.")
print(equation)

## 5. Dense neural network (ANN)

In [ ]:
# Model creation and architecture
print("\nCreating neural network model...")

# Create the model
model = create_ANN_model(X_train.shape[1], random_seed=42)

# Display model information
print_model_summary(model)

In [ ]:
# Model training
print("\nTraining the neural network...")
print("This may take a few minutes...")

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        )
    ]
)

print("\nTraining completed!")


In [ ]:
# Training history visualization
print("\nAnalyzing training progress...")

# Plot training history
plot_training_history(history)

# Print training summary
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
final_train_mae = history.history['mae'][-1]
final_val_mae = history.history['val_mae'][-1]

print(f"Final Training Metrics:")
print(f"- Training Loss: {final_train_loss:.4f}")
print(f"- Validation Loss: {final_val_loss:.4f}")
print(f"- Training MAE: {final_train_mae:.4f}")
print(f"- Validation MAE: {final_val_mae:.4f}")

In [ ]:
# Model evaluation
print("\nEvaluating model performance on test data...")


# Make predictions
y_pred = model.predict(X_test, verbose=0).flatten()

y_pred_invers = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
y_test_invers = target_scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculate evaluation metrics
metrics = evaluate_model(y_test_invers, y_pred_invers)

print(f"Test Set Performance:")
print(f"- Mean Squared Error (MSE): {metrics['mse']:.4f}")
print(f"- Root Mean Squared Error (RMSE): {metrics['rmse']:.4f}")
print(f"- Mean Absolute Error (MAE): {metrics['mae']:.4f}")
print(f"- R² Score: {metrics['r2']:.4f}")

# Plot prediction results
plot_prediction_results(y_test_invers, y_pred_invers, "Neural Network Magnitude Prediction", save_title='ANN')


## 6. Convolutional neural network (CNN)

The CNN uses complete waveforms with an explicit single-channel dimension.

In [ ]:
# Standardize each waveform sample position using training events only.
waveform_scaler = StandardScaler().fit(waveforms[train_idx])
waveforms_scaled = waveform_scaler.transform(waveforms).astype(np.float32)

# Conv1D expects (events, time samples, channels).
X_train = waveforms_scaled[train_idx, :, None]
X_val = waveforms_scaled[val_idx, :, None]
X_test = waveforms_scaled[test_idx, :, None]
y_train, y_val, y_test = (magnitudes_scaled[idx] for idx in (train_idx, val_idx, test_idx))

print("CNN training input shape:", X_train.shape)
model = create_1d_cnn_model(
    (waveforms.shape[1], 1), model_complexity='simple', random_seed=RANDOM_SEED
)
print_model_summary(model)


In [ ]:
# Model training
print("\nTraining the neural network...")
print("This may take a few minutes...")


# Train the model
history = model.fit(
    X_train, y_train,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        )
    ]
)

print("\nTraining completed!")


In [ ]:
# Training history visualization
print("\nAnalyzing training progress...")

# Plot training history
plot_training_history(history)

# Print training summary
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
final_train_mae = history.history['mae'][-1]
final_val_mae = history.history['val_mae'][-1]

print(f"Final Training Metrics:")
print(f"- Training Loss: {final_train_loss:.4f}")
print(f"- Validation Loss: {final_val_loss:.4f}")
print(f"- Training MAE: {final_train_mae:.4f}")
print(f"- Validation MAE: {final_val_mae:.4f}")

In [ ]:
# Model evaluation
print("\nEvaluating model performance on test data...")

# Make predictions
y_pred = model.predict(X_test, verbose=0).flatten()

y_pred_invers = target_scaler.inverse_transform(y_pred.reshape(-1, 1))
y_test_invers = target_scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculate evaluation metrics
metrics = evaluate_model(y_test_invers, y_pred_invers)

print(f"Test Set Performance:")
print(f"- Mean Squared Error (MSE): {metrics['mse']:.4f}")
print(f"- Root Mean Squared Error (RMSE): {metrics['rmse']:.4f}")
print(f"- Mean Absolute Error (MAE): {metrics['mae']:.4f}")
print(f"- R² Score: {metrics['r2']:.4f}")

# Plot prediction results
plot_prediction_results(y_test_invers, y_pred_invers, "Convolutional Neural Network Magnitude Prediction", save_title='CNN')


## 7. Unsupervised exploration

These clustering plots use the full dataset for exploration. Magnitudes are used only to inspect the clusters, not to fit K-means. This is separate from the held-out supervised evaluation.

In [ ]:
# Unsupervised clustering
print("\nUnsupervised clustering of features...")

from sklearn.cluster import KMeans, SpectralClustering

# Number of clusters
n_clusters = 2

# Kmeans clustering 
kmeans = KMeans(n_clusters=n_clusters, n_init='auto', random_state=42)
clusters = kmeans.fit_predict(features_scaled)

# Spectral clustering 
# spec = SpectralClustering(n_clusters=n_clusters, assign_labels='discretize', random_state=42)
# clusters = spec.fit_predict(features_scaled)

# Plot clustering results
plt.figure(figsize=(10,4))
plt.scatter(np.arange(features_scaled.shape[0]), magnitudes, 
            c = clusters/n_clusters, s = (1+clusters)*5)
plt.ylabel('Magnitude')
plt.xlabel('Event')
plt.title('Clustering of features')
plt.savefig('Clustering_features.png', dpi=300)
plt.show()


In [ ]:
# Unsupervised clustering
print("\nUnsupervised clustering of waveforms...")

from sklearn.cluster import KMeans, SpectralClustering

# Number of clusters
n_clusters = 2

# Kmeans clustering 
kmeans = KMeans(n_clusters=n_clusters, n_init='auto', random_state=42)
clusters = kmeans.fit_predict(waveforms_scaled)

# Spectral clustering 
# spec = SpectralClustering(n_clusters=n_clusters, assign_labels='discretize', random_state=42)
# clusters = spec.fit_predict(waveforms_scaled)

# Plot clustering results
plt.figure(figsize=(10,4))
plt.scatter(np.arange(waveforms_scaled.shape[0]), magnitudes, 
            c = clusters/n_clusters, s = (1+clusters)*5)
plt.ylabel('Magnitude')
plt.xlabel('Event')
plt.title('Clustering of waveforms')
plt.savefig('Clustering_waveforms.png', dpi=300)
plt.show()


## Discussion

- How do the feature models compare with the waveform CNN on the same test events?
- What would happen if you normalized each waveform by its own peak amplitude?
- How might the source-distance dependence and magnitude-dependent noise in this generator affect the learned relationships?
- Why are correlation rankings not a measure of causal feature importance?
- What additional validation would be needed before applying these models to real earthquake recordings?
